In [3]:
import pandas as pd
import numpy as np
import torch
from sklearn import preprocessing
from dgl.data import DGLDataset
import dgl
import time
import networkx as nx
import category_encoders as ce
import torch.nn as nn
import torch.nn.functional as F
import dgl.function as fn
import torch
import tqdm
import math

from typing import *
from sklearn.preprocessing import StandardScaler, Normalizer
import socket
import struct
import random
from sklearn.model_selection import train_test_split
import gc

import torch.optim as optim
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import IsolationForest
import itertools
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

graphs, _ = dgl.load_graphs('train.bin')
train_g = graphs[0]
train_g = train_g.to(device)

class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
      super(SAGELayer, self).__init__()
      self.W_apply = nn.Linear(ndim_in + ndim_out , ndim_out)
      self.activation = F.relu
      self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
      self.W_edge = nn.Linear(ndim_out * 2 + edims, 64)
      self.reset_parameters()

    def reset_parameters(self):
      gain = nn.init.calculate_gain('relu')
      nn.init.xavier_uniform_(self.W_apply.weight, gain=gain)

    def message_func(self, edges):
      # return {'m':  edges.data['h']}
        return {'m': self.W_msg(torch.cat([edges.data['h'], edges.dst['h']], 2))}

    def forward(self, g_dgl, nfeats, efeats):
      with g_dgl.local_scope():
        g = g_dgl
        g.ndata['h'] = nfeats
        g.edata['h'] = efeats
        g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
        g.ndata['h'] = F.relu(self.W_apply(torch.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))
        u, v = g.edges()
        src_feat = g.srcdata['h'][u]  # shape: [num_edges, feature_dim]
        dst_feat = g.dstdata['h'][v]  # shape: [num_edges, feature_dim]
        edge_feat = efeats      # shape: [num_edges, edge_feature_dim]

        # 拼接：[源节点特征，边特征，目标节点特征]
        edge_fin = torch.cat((src_feat, edge_feat, dst_feat), dim=2)

        # （如果你要喂进一个线性层，比如 self.W_edge）
        edge = self.W_edge(edge_fin)
        # u, v = g.edges()
        # edge = self.W_edge(torch.cat((g.srcdata['h'][u], g.dstdata['h'][v]), 2))
        return g.ndata['h'], edge
  
    
class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim,  activation):
      super(SAGE, self).__init__()
      self.atten = nn.Parameter(torch.randn(1, edim))
      self.layers = nn.ModuleList()
      self.layers.append(SAGELayer(ndim_in, edim, ndim_out, F.relu))

    def forward(self, g, nfeats, efeats, corrupt=False):
      efeats = efeats * self.atten
      if corrupt:
        e_perm = torch.randperm(g.number_of_edges())
        #n_perm = torch.randperm(g.number_of_nodes())
        efeats = efeats[e_perm]
        #nfeats = nfeats[n_perm]
      for i, layer in enumerate(self.layers):
        #nfeats = layer(g, nfeats, efeats)
        nfeats, e_feats = layer(g, nfeats, efeats)
      #return nfeats.sum(1)
      return nfeats.sum(1), e_feats.sum(1)
    
class Discriminator(nn.Module):
    def __init__(self, n_hidden):
      super(Discriminator, self).__init__()
#       self.weight = nn.Parameter(torch.Tensor(n_hidden, n_hidden))
#       self.reset_parameters()

#     def uniform(self, size, tensor):
#       bound = 1.0 / math.sqrt(size)
#       if tensor is not None:
#         tensor.data.uniform_(-bound, bound)

#     def reset_parameters(self):
#       size = self.weight.size(0)
#       self.uniform(size, self.weight)

    def forward(self, features, summary):
      features = torch.matmul(features, summary)
      return features
    
class DGI(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation):
      super(DGI, self).__init__()
      self.encoder = SAGE(ndim_in, ndim_out, edim,  F.relu)
      self.discriminator = Discriminator(ndim_out)
      self.loss = nn.BCEWithLogitsLoss()

    def forward(self, g, n_features, e_features):
      positive = self.encoder(g, n_features, e_features, corrupt=False)
      negative = self.encoder(g, n_features, e_features, corrupt=True)

      positive = positive[1]
      negative = negative[1]

      summary = torch.sigmoid(positive.mean(dim=0))

      positive = self.discriminator(positive, summary)
      negative = self.discriminator(negative, summary)

      l1 = self.loss(positive,  torch.full_like(positive, 100))
      l2 = self.loss(negative,  torch.full_like(negative, 1))

      return l1 + l2
    
ndim_in = train_g.ndata['h'].shape[2]
# hidden_features = 128
ndim_out = 128
# num_layers = 1
edim = train_g.edata['h'].shape[2]
learning_rate = 1e-3
# epochs = 4000

dgi = DGI(ndim_in,
    ndim_out,
    edim,
    F.relu).to(device)

dgi_optimizer = torch.optim.Adam(dgi.parameters(),
                lr=1e-3,
                weight_decay=0.)

epochs = 41
cnt_wait = 0
best = 1e9
best_t = 0
dur = []
node_features = train_g.ndata['h']
edge_features = train_g.edata['h']

dgi.train()

for epoch in range(epochs):
    
    
    t0 = time.time()

    dgi_optimizer.zero_grad()
    loss = dgi(train_g, node_features, edge_features)
    loss.backward()
    dgi_optimizer.step()

#     if loss < best:
#         best = loss
#         best_t = epoch
#         cnt_wait = 0
#         torch.save(dgi.state_dict(), 'best_dgi.pkl')
#     else:
#         cnt_wait += 1

#     if cnt_wait == patience:
#       print('Early stopping!')
#       break

   
    dur.append(time.time() - t0)

    if epoch % 50 == 0:

        print("Epoch {:05d} | Time(s) {:.4f} | Loss {:.4f} ".format(epoch, np.mean(dur),
              loss.item()))

Epoch 00000 | Time(s) 0.2270 | Loss -295.0244 


In [4]:
file_name = "nf-cic-v2-0.02_004.csv"
data = pd.read_csv(file_name)

lab_enc = preprocessing.LabelEncoder()
lab_enc.fit(data["Attack"])

training_emb = dgi.encoder(train_g, train_g.ndata['h'], train_g.edata['h'])[1]
training_emb = training_emb.detach().cpu().numpy()

graphs, _ = dgl.load_graphs('test.bin')
test_g = graphs[0]
test_g = test_g.to(device)

testing_emb = dgi.encoder(test_g, test_g.ndata['h'], test_g.edata['h'])[1]
testing_emb = testing_emb.detach().cpu().numpy()

df_train = pd.DataFrame(training_emb, )
df_train["Attack"] = lab_enc.inverse_transform(
        train_g.edata['Attack'].detach().cpu().numpy())
df_train["Label"] = train_g.edata['Label'].detach().cpu().numpy()

df_test = pd.DataFrame(testing_emb, )
df_test["Attack"] = lab_enc.inverse_transform(
        test_g.edata['Attack'].detach().cpu().numpy())
df_test["Label"] = test_g.edata['Label'].detach().cpu().numpy()

benign_train_samples = df_train[df_train.Label == 0].drop(columns=["Label", "Attack"])
normal_train_samples = df_train.drop(columns=["Label", "Attack"])

train_labels = df_train["Label"]
test_labels = df_test["Label"]

test_samples = df_test.drop(columns=["Label", "Attack"])

contamination = [ 0.01, 0.04]

from pyod.models.cblof import CBLOF
n_est = [2]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = CBLOF(n_clusters=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 2/2 [00:09<00:00,  4.76s/it]

{'n_estimators': 2, 'con': 0.01}
0.4877288384423759
              precision    recall  f1-score   support

           0     0.9604    0.9894    0.9747    199328
           1     0.0019    0.0005    0.0008      8130

    accuracy                         0.9506    207458
   macro avg     0.4812    0.4949    0.4877    207458
weighted avg     0.9229    0.9506    0.9365    207458



In [5]:
n_est = [3,5]
contamination = [ 0.01, 0.04]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = CBLOF(n_clusters=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:28<00:00,  7.24s/it]

{'n_estimators': 5, 'con': 0.01}
0.5075756739031643
              precision    recall  f1-score   support

           0     0.9614    0.9892    0.9751    199328
           1     0.0891    0.0258    0.0400      8130

    accuracy                         0.9515    207458
   macro avg     0.5252    0.5075    0.5076    207458
weighted avg     0.9272    0.9515    0.9385    207458



In [6]:
from pyod.models.hbos import HBOS

n_est = [5,10]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = HBOS(n_bins=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:13<00:00,  3.26s/it]


{'n_estimators': 10, 'con': 0.04}
0.5940051002273792
              precision    recall  f1-score   support

           0     0.9691    0.9601    0.9645    199328
           1     0.2027    0.2490    0.2235      8130

    accuracy                         0.9322    207458
   macro avg     0.5859    0.6045    0.5940    207458
weighted avg     0.9390    0.9322    0.9355    207458



In [7]:
n_est = [5,10]
# contamination = [0.001, 0.01, 0.04, 0.05, 0.1, 0.2]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = HBOS(n_bins=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:12<00:00,  3.08s/it]


{'n_estimators': 10, 'con': 0.04}
0.49981373596439055
              precision    recall  f1-score   support

           0     0.9608    0.9605    0.9607    199328
           1     0.0388    0.0391    0.0390      8130

    accuracy                         0.9244    207458
   macro avg     0.4998    0.4998    0.4998    207458
weighted avg     0.9247    0.9244    0.9245    207458



In [8]:
from pyod.models.pca import PCA
n_est = [5,10]
cont = [0.01, 0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = PCA(n_components=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:13<00:00,  3.38s/it]

{'n_estimators': 5, 'con': 0.01}
0.48772753351689946
              precision    recall  f1-score   support

           0     0.9604    0.9894    0.9747    199328
           1     0.0019    0.0005    0.0008      8130

    accuracy                         0.9506    207458
   macro avg     0.4812    0.4949    0.4877    207458
weighted avg     0.9229    0.9506    0.9365    207458



In [9]:
n_est = [5,10]
cont = [0.01, 0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = PCA(n_components=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:13<00:00,  3.47s/it]


{'n_estimators': 5, 'con': 0.01}
0.48761531401095276
              precision    recall  f1-score   support

           0     0.9604    0.9889    0.9745    199328
           1     0.0018    0.0005    0.0008      8130

    accuracy                         0.9502    207458
   macro avg     0.4811    0.4947    0.4876    207458
weighted avg     0.9228    0.9502    0.9363    207458



In [10]:
from sklearn.ensemble import IsolationForest
n_est = [20, 50]
cont = [0.01,0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = IsolationForest(n_estimators=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = list(map(lambda x : 0 if x == 1 else 1, y_pred))

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:24<00:00,  6.15s/it]


{'n_estimators': 50, 'con': 0.04}
0.4951306130436025
              precision    recall  f1-score   support

           0     0.9604    0.9570    0.9587    199328
           1     0.0303    0.0330    0.0316      8130

    accuracy                         0.9208    207458
   macro avg     0.4954    0.4950    0.4951    207458
weighted avg     0.9240    0.9208    0.9224    207458



In [11]:
from sklearn.ensemble import IsolationForest

n_est = [20, 50]
cont = [ 0.01,0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = IsolationForest(n_estimators=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = list(map(lambda x : 0 if x == 1 else 1, y_pred))

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:26<00:00,  6.58s/it]


{'n_estimators': 50, 'con': 0.04}
0.4951619598863207
              precision    recall  f1-score   support

           0     0.9604    0.9570    0.9587    199328
           1     0.0303    0.0330    0.0316      8130

    accuracy                         0.9208    207458
   macro avg     0.4954    0.4950    0.4952    207458
weighted avg     0.9240    0.9208    0.9224    207458

